# Privacy Lens · 이미지 객체(번호판·얼굴) 검증·학습

D-16: YOLO 는 글자가 아닌 이미지 객체를 맡는다. 이 노트북은 **직접 학습 전에 공개 모델이 쓸 만한지** 재고, 필요하면 번호판을 학습한다.

| 칸 | 하는 일 | 시간(T4) |
|---|---|---|
| 0 → 1 | 설정·설치 | 3~5분 |
| 2 | 공개 번호판 모델(ONNX) 받기 + 해시 기록 | 1분 |
| 3 | Open Images 에서 번호판·얼굴 사진만 받아 YOLO 형식으로 | 5~15분 |
| 4 | 공개 번호판 모델 / YuNet 얼굴 검출을 **같은 Open Images 검증 사진**으로 재기 | 2~5분 |
| 5 | (선택) YOLO26n 으로 번호판 직접 학습(Open Images test 분할 사진) → 4번과 같은 검증 사진으로 비교 | 20~40분 |
| 6 | 결과를 드라이브에 저장 | 즉시 |

런타임 → 런타임 유형 변경 → **T4 GPU**. 받는 사진은 Open Images(라벨 CC BY 4.0)이고 드라이브의 `PrivacyLens-YOLO/objects/` 에만 둔다(저장소 금지).
**주의:** Open Images 숫자는 해외 사진 기준이다. 한국 번호판·휴대폰 사진은 실사 평가 폴더로 따로 잰다.

## 0. 설정

In [ ]:
ROOT = "/content/drive/MyDrive/PrivacyLens-YOLO"   # 결과·모델을 둘 곳(공유 폴더 바로가기)
BRANCH = "feat/vision-objects"
PLATE_REPO = "ishaqinu/yolo26n-license-plate-detection"
PLATE_FILE = "alpr_26n.onnx"     # .pt 는 pickle 이라 열면 코드가 실행될 수 있다. ONNX 만 받는다
VAL_N = 300                      # 검증 사진 수(번호판·얼굴 각각)
TRAIN_N = 2000                   # 5번 칸 학습 사진 수
EPOCHS = 30

## 1. 드라이브 연결 · 코드 받기 · 설치

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, json, hashlib, pathlib
OUT = f"{ROOT}/objects"; os.makedirs(OUT, exist_ok=True)
if not os.path.isdir("/content/privacy-lens"):
    subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/2026-usw-project/privacy-lens", "/content/privacy-lens"], check=True)
%cd /content/privacy-lens
!git fetch -q origin && git checkout -q $BRANCH && git pull -q
!pip -q install -r requirements.txt "ultralytics>=8.4" onnxruntime-gpu huggingface_hub fiftyone
!python -m pipeline.objects download
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음(CPU)")

## 2. 공개 번호판 모델 받기

받은 파일의 커밋(revision)과 SHA-256 을 기록한다. 같은 파일인지 나중에 확인할 수 있게.

In [ ]:
from huggingface_hub import hf_hub_download, HfApi
rev = HfApi().model_info(PLATE_REPO).sha
plate_path = hf_hub_download(PLATE_REPO, PLATE_FILE, revision=rev)
digest = hashlib.sha256(open(plate_path, "rb").read()).hexdigest()
print("revision", rev, "\nsha256", digest, "\n", plate_path)
json.dump({"repo": PLATE_REPO, "file": PLATE_FILE, "revision": rev, "sha256": digest},
          open(f"{OUT}/plate_model_source.json", "w"), indent=1)

## 3. Open Images 에서 번호판·얼굴 사진만 받기

FiftyOne 으로 해당 클래스가 있는 사진만 받는다(전체 561GB 를 받지 않는다).

In [ ]:
import fiftyone as fo, fiftyone.zoo as foz
DATA = "/content/oi"

def export(cls, split, n, name):
    ds = foz.load_zoo_dataset("open-images-v7", split=split, label_types=["detections"], classes=[cls],
                              max_samples=n, only_matching=True, seed=0, shuffle=True,
                              dataset_name=f"{name}-{split}-{n}")
    ds.export(export_dir=f"{DATA}/{name}", dataset_type=fo.types.YOLOv5Dataset, label_field="ground_truth",
              split="val" if split == "validation" else "train", classes=[cls])
    ds.delete()  # 메모리 아끼기
    return len(ds)

print("번호판 검증", export("Vehicle registration plate", "validation", VAL_N, "plate"))
print("얼굴 검증", export("Human face", "validation", VAL_N, "face"))

## 4. 공개 모델 재기 — 같은 Open Images 검증 사진

In [ ]:
from ultralytics import YOLO
import yaml
yp = f"{DATA}/plate/dataset.yaml"
cfg = yaml.safe_load(open(yp)); cfg["path"] = f"{DATA}/plate"
cfg.setdefault("train", cfg["val"])  # 검증만 받았을 때도 YAML 형식을 맞춘다(학습은 5번 칸)
yaml.safe_dump(cfg, open(yp, "w"))
r = YOLO(plate_path, task="detect").val(data=yp, split="val", imgsz=640, batch=1, verbose=False, plots=False)
plate_pub = {"mAP50": round(float(r.box.map50), 4), "mAP50_95": round(float(r.box.map), 4),
             "precision": round(float(r.box.mp), 4), "recall": round(float(r.box.mr), 4)}
print("공개 번호판 모델(Open Images 검증):", plate_pub)

# YuNet: 정답 얼굴(20px 이상)을 IoU 0.5 이상으로 맞힌 비율
from PIL import Image
from pipeline.objects import FaceDetector, _iou, MODELS, YUNET_FILE
from pipeline.types import Box
fd = FaceDetector(MODELS / YUNET_FILE)
tp = fn = fp = 0
for img in sorted(pathlib.Path(f"{DATA}/face/images/val").glob("*")):
    im = Image.open(img).convert("RGB"); W, H = im.size
    lab = pathlib.Path(str(img).replace("/images/", "/labels/")).with_suffix(".txt")
    gt = []
    for line in (lab.read_text().splitlines() if lab.exists() else []):
        _, cx, cy, w, h = map(float, line.split())
        b = Box(int((cx - w / 2) * W), int((cy - h / 2) * H), int(w * W), int(h * H))
        if b.w >= 20 and b.h >= 20:
            gt.append(b)
    pred = [o.box for o in fd.detect(im)]
    used = set()
    for g in gt:
        j = next((k for k, p in enumerate(pred) if k not in used and _iou(g, p) >= 0.5), None)
        if j is None: fn += 1
        else: tp += 1; used.add(j)
    fp += len(pred) - len(used)
face_pub = {"recall": round(tp / max(1, tp + fn), 4), "precision": round(tp / max(1, tp + fp), 4),
            "faces": tp + fn}
print("YuNet 얼굴(20px 이상):", face_pub)

## 5. (선택) 번호판 직접 학습 — YOLO26n

4번과 **같은 검증 사진**으로 잰다. 공개 모델보다 낫지 않으면 공개 모델을 쓴다.

In [ ]:
# Open Images 의 train 분할은 라벨 파일만 수 GB 라 무료 Colab 메모리(12GB)를 넘어 세션이 죽었다(2026-11-02).
# 그래서 test 분할(검증과 겹치지 않음)에서 학습 사진을 받는다.
print("번호판 학습", export("Vehicle registration plate", "test", TRAIN_N, "plate"))
cfg = yaml.safe_load(open(yp)); cfg["path"] = f"{DATA}/plate"
cfg.setdefault("train", cfg["val"])  # 검증만 받았을 때도 YAML 형식을 맞춘다(학습은 5번 칸)
yaml.safe_dump(cfg, open(yp, "w"))
m = YOLO("yolo26n.pt")
m.train(data=yp, epochs=EPOCHS, imgsz=640, batch=16, workers=2, project="/content/runs", name="plate", exist_ok=True,
        patience=10, verbose=False, plots=False)
best = "/content/runs/plate/weights/best.pt"
r = YOLO(best).val(data=yp, split="val", imgsz=640, verbose=False, plots=False)
plate_ours = {"mAP50": round(float(r.box.map50), 4), "mAP50_95": round(float(r.box.map), 4),
              "precision": round(float(r.box.mp), 4), "recall": round(float(r.box.mr), 4)}
print("우리 학습(같은 검증 사진):", plate_ours, "\n공개 모델:", plate_pub)
os.makedirs(f"{OUT}/plate_v001", exist_ok=True)
YOLO(best).export(format="onnx", imgsz=640)
!cp /content/runs/plate/weights/best.pt /content/runs/plate/weights/best.onnx "$OUT/plate_v001/"

## 6. 결과 저장

In [ ]:
import datetime
res = {"when": datetime.datetime.now().isoformat(timespec="minutes"), "val_images": VAL_N,
       "plate_public": plate_pub, "face_yunet": face_pub,
       "plate_ours": globals().get("plate_ours"), "train_images": TRAIN_N if "plate_ours" in globals() else 0}
json.dump(res, open(f"{OUT}/objects_eval.json", "w"), ensure_ascii=False, indent=1)
print(json.dumps(res, ensure_ascii=False, indent=1))